# TFM_00_Master_Template: Plantilla de Control Multi-Planta con RL y MLflow

**Objetivo:** Este notebook contiene la estructura base para el entrenamiento de agentes de Reinforcement Learning (RL) en el entorno `ThreePlantEnv`, asegurando el registro sistemático de experimentos y métricas mediante **MLflow**.

## I. Configuración del Entorno y Librerías

In [ ]:
# ====================================================================
# INSTALACIÓN E IMPORTACIONES
# ====================================================================
#%pip install stable-baselines3[extra] gymnasium matplotlib pandas shimmy mlflow

import os
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
import pandas as pd

# Librerías de RL
from stable_baselines3 import PPO, TD3, SAC

# MLflow
import mlflow
from mlflow.tracking import MlflowClient

# Configuraciones de Matplotlib (opcional)
plt.style.use('ggplot')

## II. Configuración y Setup de MLflow (Solo Ejecutar una Vez)

In [ ]:
# --------------------------------------------------------------------
# CONFIGURACIÓN DEL EXPERIMENTO MLflow (Sección II - CORREGIDA)
# --------------------------------------------------------------------
EXPERIMENT_NAME = "TFM_Control_MultiPlanta_RL"
TRACKING_URI = "sqlite:///mlflow.db" 

mlflow.set_tracking_uri(TRACKING_URI)
try:
    # Intenta obtener el ID del experimento si ya existe
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
    print(f"Usando experimento existente: {EXPERIMENT_NAME}")
except:
    # Si no existe, crea uno nuevo (devuelve el ID como string)
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print(f"Creando nuevo experimento: {EXPERIMENT_NAME}")

# Establece el ID del experimento para todas las llamadas subsiguientes
mlflow.set_experiment(experiment_id=experiment_id) 
print(f"Contexto MLflow activo en ID: {experiment_id}")

# Directorio local para guardar los modelos antes de subirlos como artefactos
LOCAL_ARTIFACTS_DIR = "mlflow_local_artifacts"
os.makedirs(LOCAL_ARTIFACTS_DIR, exist_ok=True)
print(f"Artefactos locales guardados en: {LOCAL_ARTIFACTS_DIR}")

## III. Definición del Entorno (`ThreePlantEnv`)

In [ ]:
# --------------------------------------------------------------------
# 1. CLASE MODELO FOPDT
# --------------------------------------------------------------------
class FOPDT_Plant:
    def __init__(self, Kp, Tp, Theta):
        self.Kp = Kp
        self.Tp = Tp
        self.Theta = Theta
        self.y_out = 0.0
        self.dead_time_buffer = [0.0] * int(Theta) # Inicializa el buffer de tiempo muerto

    def step(self, u, dt=1.0):
        # Añadir la nueva acción al buffer y obtener la acción retrasada
        self.dead_time_buffer.append(u)
        u_delayed = self.dead_time_buffer.pop(0)
            
        # Actualización de la salida (Euler simple)
        # dY/dt = (Kp*u_delayed - Y) / Tp  =>  Y_k = Y_{k-1} + dt * (...) 
        self.y_out += (dt / self.Tp) * (self.Kp * u_delayed - self.y_out)
        
        # Mejora Opcional: Ruido de Proceso para robustez
        self.y_out += np.random.normal(0, 0.005) 
        
        # Asegurar que la salida no sea negativa
        self.y_out = max(0.0, self.y_out)
        
        return self.y_out

# --------------------------------------------------------------------
# 2. FUNCIÓN DE GENERACIÓN DE SETPOINTS
# --------------------------------------------------------------------
def generate_complex_profile(length, min_val, max_val, min_step, max_step):
    """Genera un perfil de setpoint aleatorio con pasos y rampas."""
    profile = np.zeros(length)
    current_val = np.random.uniform(min_val, max_val)
    profile[0] = current_val
    time = 1
    while time < length:
        step_len = np.random.randint(min_step, max_step)
        next_val = np.random.uniform(min_val, max_val)
        
        # Implementación de rampa suave
        for t in range(time, min(time + step_len, length)):
            profile[t] = current_val + (next_val - current_val) * (t - time) / step_len
            
        current_val = next_val
        time += step_len
    return profile

# --------------------------------------------------------------------
# 3. ENTORNO PERSONALIZADO ThreePlantEnv (Con Error Integral)
# --------------------------------------------------------------------
class ThreePlantEnv(gym.Env):
    def __init__(self, **kwargs):
        super().__init__()
        self.CAPACITIES = np.array([30.0, 80.0, 120.0]) # MW
        self.MAX_TOTAL_CAPACITY = self.CAPACITIES.sum() # 230 MW
        
        self.plants = [
            FOPDT_Plant(Kp=1.0, Tp=15, Theta=5), 
            FOPDT_Plant(Kp=0.8, Tp=40, Theta=15),
            FOPDT_Plant(Kp=1.1, Tp=25, Theta=8)
        ]
        
        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(3,), dtype=np.float32) # u_i: 0-1, se escala a capacidad
        
        # Espacio de Observación: [Error, Error_Integral, Salida_P1, Salida_P2, Salida_P3, Setpoint]
        low = np.array([-np.inf, -np.inf, 0.0, 0.0, 0.0, 0.0])
        high = np.array([np.inf, np.inf, 1.0, 1.0, 1.0, 1.0])
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)

        self.current_setpoint_mw = 0.0 # Setpoint en MW
        self.cumulative_error = 0.0 # Estado del error integral
        self.last_actions_mw = np.zeros(3)
        self.time_step = 0
        
        # Se usará externamente para inyectar el setpoint
        self.setpoint_profile = [] 
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Re-inicializa las plantas
        self.plants = [
            FOPDT_Plant(Kp=1.0, Tp=15, Theta=5), 
            FOPDT_Plant(Kp=0.8, Tp=40, Theta=15),
            FOPDT_Plant(Kp=1.1, Tp=25, Theta=8)
        ]
        self.current_setpoint_mw = 0.0
        self.cumulative_error = 0.0
        self.last_actions_mw = np.zeros(3)
        self.time_step = 0
        
        # El setpoint se inyectará en el loop de entrenamiento/evaluación
        observation = self._get_obs()
        info = {}
        return observation, info

    def _get_obs(self):
        plant_outputs_mw = np.array([p.y_out for p in self.plants])
        current_total_output_mw = plant_outputs_mw.sum()
        
        # Normalización
        output_normalized = current_total_output_mw / self.MAX_TOTAL_CAPACITY
        setpoint_normalized = self.current_setpoint_mw / self.MAX_TOTAL_CAPACITY
        
        error = setpoint_normalized - output_normalized
        
        return np.array([
            error, 
            self.cumulative_error / 1000.0, # Normalizar el error integral (ejemplo)
            plant_outputs_mw[0] / self.CAPACITIES[0], # Salida P1 normalizada a su capacidad máxima
            plant_outputs_mw[1] / self.CAPACITIES[1],
            plant_outputs_mw[2] / self.CAPACITIES[2],
            setpoint_normalized
        ], dtype=np.float32)

    def step(self, action):
        self.time_step += 1
        
        # Escalar acción (0 a 1) a capacidad real (MW)
        actions_mw = action * self.CAPACITIES 
        
        # Actualizar plantas
        plant_outputs_mw = np.array([p.step(u) for p, u in zip(self.plants, actions_mw)])
        current_total_output_mw = plant_outputs_mw.sum()

        # Normalización para cálculo de error
        setpoint_normalized = self.current_setpoint_mw / self.MAX_TOTAL_CAPACITY
        output_normalized = current_total_output_mw / self.MAX_TOTAL_CAPACITY
        error = setpoint_normalized - output_normalized
        
        # Actualizar error integral
        self.cumulative_error += error
        
        # REWARD FUNCTION (Robusta y con Costo)
        # a) Penalización por error (principal)
        reward_tracking = -1.0 * (error ** 2)
        
        # b) Penalización por cambio de actuación (Suavidad / Desgaste)
        action_diff = np.abs(actions_mw - self.last_actions_mw) / self.CAPACITIES # Normalizar el delta
        reward_actuation = -0.05 * action_diff.sum()
        
        # c) Penalización por restricción (Si el agente intenta sobrepasar la capacidad)
        capacity_penalty = -50.0 if np.any(actions_mw > self.CAPACITIES * 1.05) else 0.0
        
        reward = reward_tracking + reward_actuation + capacity_penalty

        # Actualizar estados internos
        self.last_actions_mw = actions_mw
        
        observation = self._get_obs()
        terminated = False
        truncated = False
        info = {}
        
        return observation, reward, terminated, truncated, info

## IV. Funciones Auxiliares (Evaluación y Ploteo)

In [ ]:
def evaluate_agent_on_profile(model, env, profile):
    """Evalúa un agente RL sobre un perfil de setpoint dado, retornando métricas."""
    # Nota: Este env debe ser una copia para no afectar el env de entrenamiento
    obs, info = env.reset()
    
    total_error_sq = 0
    total_actuation_cost = 0
    total_steps = len(profile)
    
    for t in range(total_steps):
        # Inyectar el setpoint del perfil (en MW)
        env.current_setpoint_mw = profile[t] * env.MAX_TOTAL_CAPACITY
        
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        
        # Recalcular el error normalizado para la métrica RMSE (si el step lo cambia)
        current_total_output_mw = np.array([p.y_out for p in env.plants]).sum()
        error_normalized = (env.current_setpoint_mw - current_total_output_mw) / env.MAX_TOTAL_CAPACITY
        total_error_sq += error_normalized**2
        
        # Costo de actuación
        total_actuation_cost += np.abs(env.last_actions_mw - env.last_actions_mw).sum()

    rmse = np.sqrt(total_error_sq / total_steps)
    # Podrías calcular un IAE (Integral Absolute Error) o un Costo de Actuación mejor
    
    return rmse, total_actuation_cost

def rollout_and_plot(model, env, profile, title="Resultado del Agente"):
    """Ejecuta el agente y genera una gráfica de seguimiento."""
    obs, info = env.reset()
    
    # Almacenamiento de datos
    demanda_data = []
    salida_total_data = []
    acciones_data = []
    
    for t in range(len(profile)):
        env.current_setpoint_mw = profile[t] * env.MAX_TOTAL_CAPACITY
        
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        
        demanda_data.append(env.current_setpoint_mw)
        salida_total_data.append(np.array([p.y_out for p in env.plants]).sum())
        acciones_data.append(env.last_actions_mw)
    
    df_acciones = pd.DataFrame(acciones_data, columns=['u1', 'u2', 'u3'])
    
    fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    # Gráfico 1: Seguimiento del Setpoint
    ax[0].plot(demanda_data, label='Demanda (Setpoint)', color='blue', linestyle='--')
    ax[0].plot(salida_total_data, label='Salida Total', color='red')
    ax[0].set_title(f'Seguimiento de Demanda - {title}')
    ax[0].set_ylabel('Potencia (MW)')
    ax[0].legend()
    ax[0].grid(True)
    
    # Gráfico 2: Acciones (Actuadores)
    df_acciones.plot(ax=ax[1], title='Acciones de los Actuadores')
    ax[1].set_ylabel('Potencia Asignada (MW)')
    ax[1].set_xlabel('Paso de Tiempo')
    ax[1].grid(True)
    
    plt.tight_layout()
    
    return fig

## V. Bucle Maestro de Entrenamiento con Registro MLflow

In [ ]:
# --------------------------------------------------------------------
# V. BUCLE MAESTRO DE ENTRENAMIENTO CON REGISTRO MLflow (REVISADO)
# --------------------------------------------------------------------

# 1. Definición de las configuraciones a probar
AGENTS_TO_RUN = {
    # Revisa si los agentes TD3 o SAC fallan al inicializar si no están instalados.
    "PPO_V1_Base": (PPO, {"learning_rate": 3e-4, "n_steps": 2048, "gamma": 0.99}),
    "TD3_V1_Test": (TD3, {"learning_rate": 1e-3, "buffer_size": 100000, "gamma": 0.98}),
    "SAC_V1_Test": (SAC, {"learning_rate": 5e-4, "tau": 0.01, "batch_size": 64}),
}

TOTAL_TIMESTEPS = 50000 
SEED = 42

# Perfil de prueba
test_profile = generate_complex_profile(length=2000, min_val=0.2, max_val=0.8, min_step=30, max_step=120)

# 2. Bucle de Entrenamiento
for run_name, (AgentClass, params) in AGENTS_TO_RUN.items():
    
    # --- INICIO DEL RUN --- (MLflow)
    with mlflow.start_run(run_name=run_name) as run:
        
        print(f"\n🚀 Iniciando Run: {run_name}")
        
        try:
            # 3. LOGGEAR PARÁMETROS
            mlflow.log_params(params)
            mlflow.log_param("total_timesteps", TOTAL_TIMESTEPS)
            mlflow.log_param("agent_type", AgentClass.__name__)
            
            # 4. CREAR Y ENTRENAR EL AGENTE
            env = ThreePlantEnv() 
            model = AgentClass("MlpPolicy", env, verbose=0, seed=SEED, **params)
            
            # --- Entrenar el modelo (aquí puede fallar) ---
            model.learn(total_timesteps=TOTAL_TIMESTEPS)
            
            # 5. EVALUAR Y LOGGEAR MÉTRICAS FINALES
            env_eval = ThreePlantEnv()
            rmse, actuation_cost = evaluate_agent_on_profile(model, env_eval, test_profile)
            
            mlflow.log_metric("RMSE_Test", rmse)
            mlflow.log_metric("Costo_Actuacion_Test", actuation_cost)
            
            print(f"📊 Run {run_name} finalizado. RMSE: {rmse:.4f}")
            
            # 6. GUARDAR Y LOGGEAR ARTEFACTOS (Modelo)
            model_filename = f"{run_name}_model.zip"
            model_path = os.path.join(LOCAL_ARTIFACTS_DIR, model_filename)
            model.save(model_path)
            mlflow.log_artifact(model_path, "Modelos_Checkpoints")
            
            print(f"✅ Run {run_name} registrado en MLflow (ID: {run.info.run_id})")

        except Exception as e:
            # Si hay un error, lo registramos como métrica de fallo para diagnosticar
            mlflow.log_metric("Entrenamiento_Fallido", 1) 
            print(f"❌ ERROR CRÍTICO en el Run {run_name}: {e}")
            print("El run se registrará con métricas vacías para diagnóstico.")

In [ ]:
# 7. LOGGEAR GRÁFICA DE RESULTADOS (Visualización en MLflow)
fig = rollout_and_plot(model, env_eval, test_profile, title=run_name)
fig_path = os.path.join(LOCAL_ARTIFACTS_DIR, f"{run_name}_grafica.png")
fig.savefig(fig_path)
mlflow.log_artifact(fig_path, "Graficas_Resultados")
plt.close(fig) # Cierra la figura para liberar memoria
        
print(f"✅ Run {run_name} registrado en MLflow (ID: {run.info.run_id})")

## VI. Visualización y Carga de Modelos

In [ ]:
# --------------------------------------------------------------------
# Cargar y Graficar los 3 Agentes Principales (Comparativa)
# --------------------------------------------------------------------
# Asegúrate de que PPO, TD3, SAC y MlflowClient están importados al inicio del notebook.

# 1. Configuración de MLflow
TRACKING_URI = "sqlite:///mlflow.db"
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
EXPERIMENT_NAME = "TFM_Control_MultiPlanta_RL"

# 🔥 MODIFICA ESTA LISTA EN CADA ITERACIÓN (V1, V2, V3...)
# Aquí se cargan los runs de tu Línea Base V1.
AGENTS_TO_PLOT = [
    "PPO_V1_Base", 
    "TD3_V1_Test", 
    "SAC_V1_Test"
] 

# NOTA: Para graficar los runs de la optimización V2, esta lista debe ser:
# AGENTS_TO_PLOT = ["PPO_V2_MaxGamma", "TD3_V2_HighGamma_LR_5e4", "SAC_V2_Stable"] 

models_to_plot = {}
env_eval = ThreePlantEnv() 
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
experiment_id = experiment.experiment_id if experiment else None

for run_name_filter in AGENTS_TO_PLOT:
    try:
        # Buscar el Run con el nombre exacto (asumimos que la V1 ya existe)
        runs = client.search_runs(
            experiment_ids=[experiment_id],
            filter_string=f"tags.`mlflow.runName` = '{run_name_filter}'",
            order_by=["start_time DESC"],
            max_results=1
        )

        if not runs:
            print(f"⚠️ No se encontró el Run con nombre '{run_name_filter}'. Saltando.")
            continue
            
        run = runs[0]
        agent_type = run.data.params.get("agent_type")
        
        # 2. Determinar la Clase de Agente
        AGENT_CLASS = globals().get(agent_type)
        if AGENT_CLASS is None:
            raise ImportError(f"No se pudo encontrar la clase para el agente: {agent_type}")
            
        # 3. Cargar el modelo
        model_filename = f"{run_name_filter}_model.zip"
        ARTIFACT_PATH = f"Modelos_Checkpoints/{model_filename}"
        
        model_path_uri = mlflow.get_artifact_uri(ARTIFACT_PATH)
        if model_path_uri.startswith('file:/') and not model_path_uri.startswith('file:///'):
            model_path_uri = model_path_uri.replace('file:/', 'file:///')
            
        model = AGENT_CLASS.load(model_path_uri, env=env_eval)
        models_to_plot[run_name_filter] = model
        print(f"✅ Modelo {run_name_filter} cargado (RMSE: {run.data.metrics.get('RMSE_Test'):.4f})")

    except Exception as e:
        print(f"❌ ERROR al cargar {run_name_filter}: {e}")

# 4. Generar las Gráficas
if models_to_plot:
    print("\n--- Generando Gráficas Comparativas ---")
    for name, model in models_to_plot.items():
        # Recuperamos el RMSE exacto para el título
        run_data = client.search_runs(
            experiment_ids=[experiment_id],
            filter_string=f"tags.`mlflow.runName` = '{name}'",
            order_by=["start_time DESC"],
            max_results=1
        )[0]
        rmse_value = run_data.data.metrics.get('RMSE_Test', 0.0)
        
        fig = rollout_and_plot(
            model, 
            env_eval, 
            test_profile, 
            title=f"Resultados de {name} - RMSE: {rmse_value:.4f}"
        )
        fig.show()
else:
    print("No se cargó ningún modelo. Asegúrate de que los runs V1/V2 están completos.")